# Exercise 17.1 solution


## Exercise 17.1a solution


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# Parameters
k = 2.0
A = 1.0
alpha = 0.1
L = 100
eps = 0.005
gamma = 2.0

dx = 1
dt = 0.1
N = int(L / dx)
n_steps = 1400
save_every = 10

# Initialize arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)
left = int(N / 10)
v[:left] = 0.3

I = np.arange(1, N)
Ip = I + 1
Im = I - 1

snapshots_v = [v.copy()]
snapshots_w = [w.copy()]

for i in range(n_steps):
    I_ion = A * v * (1 - v) * (v - alpha) - w

    # First add the diffusion to v
    v[I] = v[I] + dt * (k / dx**2) * (v[Ip] - 2 * v[I] + v[Im])
    v[0] = v[0] + dt * (k / dx**2) * 2 * (v[1] - v[0])
    v[N] = v[N] + dt * (k / dx**2) * 2 * (v[N - 1] - v[N])

    # Then the reaction terms
    v = v + dt * I_ion
    w = w + dt * eps * (v - gamma * w)

    if (i + 1) % save_every == 0:
        snapshots_v.append(v.copy())
        snapshots_w.append(w.copy())

snapshots_v = np.array(snapshots_v)
snapshots_w = np.array(snapshots_w)
x = np.linspace(0, L, N + 1)


def plot_fhn(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots_v[frame], color="C0", linewidth=2, label="Voltage (V)")
    ax.plot(
        x,
        snapshots_w[frame],
        color="C1",
        linewidth=2,
        linestyle="--",
        label="Recovery (w)",
    )
    ax.set_xlim(0, L)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("V / w")
    t_ms = frame * save_every * dt
    ax.set_title(f"FHN Model — t = {t_ms:.1f} ms")
    ax.legend(loc="upper right")
    plt.show()


interact(
    plot_fhn,
    frame=IntSlider(
        min=0, max=len(snapshots_v) - 1, step=1, value=0, description="Time frame"
    ),
)

interactive(children=(IntSlider(value=0, description='Time frame', max=140), Output()), _dom_classes=('widget-…

<function __main__.plot_fhn(frame=0)>

**Reflection answer:** Unlike the bistable wave (which leaves the cable permanently at $V=1$), the FHN wave is a _pulse_: behind the wavefront, the voltage drops back toward 0. This happens because the recovery variable $w$ slowly increases in the activated region, eventually overwhelming the cubic reaction term and forcing repolarization. The result is a travelling pulse rather than a travelling front.


## Exercise 17.1b solution


In [ ]:
# Reset arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Stimulus in the middle
mid = int(N / 2)
v[mid - 10 : mid + 10] = 0.3

# Wrap-around slicing
I_all = np.arange(N + 1)
Ip_all = I_all + 1
Ip_all[N] = 0
Im_all = I_all - 1
Im_all[0] = N

n_steps_p = 1400
snapshots_v_p = [v.copy()]
snapshots_w_p = [w.copy()]

for i in range(n_steps_p):
    I_ion = A * v * (1 - v) * (v - alpha) - w
    v[I_all] = v[I_all] + dt * (k / dx**2) * (v[Ip_all] - 2 * v[I_all] + v[Im_all])
    v = v + dt * I_ion
    w = w + dt * eps * (v - gamma * w)

    if (i + 1) % save_every == 0:
        snapshots_v_p.append(v.copy())
        snapshots_w_p.append(w.copy())

snapshots_v_p = np.array(snapshots_v_p)
snapshots_w_p = np.array(snapshots_w_p)


def plot_periodic(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots_v_p[frame], color="C3", linewidth=2, label="Voltage")
    ax.plot(
        x,
        snapshots_w_p[frame],
        color="C4",
        linewidth=2,
        linestyle="--",
        label="Recovery",
    )
    ax.set_xlim(0, L)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("V / w")
    t_ms = frame * save_every * dt
    ax.set_title(f"Periodic boundaries — t = {t_ms:.1f} ms")
    ax.legend(loc="upper right")
    plt.show()


interact(
    plot_periodic,
    frame=IntSlider(
        min=0, max=len(snapshots_v_p) - 1, step=1, value=0, description="Time frame"
    ),
)

interactive(children=(IntSlider(value=0, description='Time frame', max=140), Output()), _dom_classes=('widget-…

<function __main__.plot_periodic(frame=0)>

**Answer:** The two wavefronts collide and annihilate each other. This happens because each wavefront runs into the _refractory tail_ of the other — the region where $w$ is still elevated after the passage of the action potential. Since the tissue is refractory, neither wavefront can propagate into the region occupied by the other, and both die out.


## Exercise 17.1c solution


In [ ]:
# Reset arrays
v = np.zeros(N + 1)
w = np.zeros(N + 1)

# Asymmetric initial conditions for reentry
mid = int(N / 2)
v[mid - 10 : mid + 10] = 0.3
w[: mid - 5] = 0.2  # Left side is refractory

# Same periodic slicing
I_all = np.arange(N + 1)
Ip_all = I_all + 1
Ip_all[N] = 0
Im_all = I_all - 1
Im_all[0] = N

n_steps_reentry = 14000
save_every_r = 50
snapshots_v_r = [v.copy()]
snapshots_w_r = [w.copy()]

for i in range(n_steps_reentry):
    I_ion = A * v * (1 - v) * (v - alpha) - w
    v[I_all] = v[I_all] + dt * (k / dx**2) * (v[Ip_all] - 2 * v[I_all] + v[Im_all])
    v = v + dt * I_ion
    w = w + dt * eps * (v - gamma * w)

    if (i + 1) % save_every_r == 0:
        snapshots_v_r.append(v.copy())
        snapshots_w_r.append(w.copy())

snapshots_v_r = np.array(snapshots_v_r)
snapshots_w_r = np.array(snapshots_w_r)


def plot_reentry(frame=0):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, snapshots_v_r[frame], color="C3", linewidth=2, label="Voltage")
    ax.plot(
        x,
        snapshots_w_r[frame],
        color="C4",
        linewidth=2,
        linestyle="--",
        label="Recovery",
    )
    ax.set_xlim(0, L)
    ax.set_ylim(-0.2, 1.2)
    ax.set_xlabel("Position (x)")
    ax.set_ylabel("V / w")
    t_ms = frame * save_every_r * dt
    ax.set_title(f"Reentry — t = {t_ms:.1f} ms")
    ax.legend(loc="upper right")
    plt.show()


interact(
    plot_reentry,
    frame=IntSlider(
        min=0, max=len(snapshots_v_r) - 1, step=1, value=0, description="Time frame"
    ),
)

interactive(children=(IntSlider(value=0, description='Time frame', max=280), Output()), _dom_classes=('widget-…

<function __main__.plot_reentry(frame=0)>

**Answer:** With the asymmetric initial condition, the wavefront travelling to the left is blocked by the refractory tissue ($w = 0.2$), while the rightward wave propagates freely. When the rightward wave wraps around the ring (via periodic BCs) and returns to the starting region, the tissue has had time to recover ($w$ has decayed back toward 0). The wave can therefore re-excite the tissue, creating a self-sustaining reentrant circuit that keeps circulating indefinitely.

This is the mechanism behind many cardiac arrhythmias: _unidirectional block_ combined with a _circuit long enough for recovery_ allows the electrical wave to chase its own tail.


## Exercise 17.1d solution


In [ ]:
from ipywidgets import interact, FloatSlider


def fhn_phase_plane(alpha_w=0.1, eps_w=0.005, gamma_w=2.0, V0=0.15):
    """Simulate a single FHN cell and show phase plane + time traces."""
    A_w = 1.0
    dt_w = 0.05
    T_total = 600.0  # ms
    n = int(T_total / dt_w)

    # Simulate the ODE
    V_trace = np.zeros(n)
    w_trace = np.zeros(n)
    V_trace[0] = V0
    w_trace[0] = 0.0

    for i in range(1, n):
        V_old = V_trace[i - 1]
        w_old = w_trace[i - 1]
        dV = A_w * V_old * (1 - V_old) * (V_old - alpha_w) - w_old
        dw = eps_w * (V_old - gamma_w * w_old)
        V_trace[i] = V_old + dt_w * dV
        w_trace[i] = w_old + dt_w * dw

    # Nullclines
    V_nc = np.linspace(-0.2, 1.2, 300)
    w_V_nullcline = A_w * V_nc * (1 - V_nc) * (V_nc - alpha_w)  # dV/dt = 0
    w_w_nullcline = V_nc / gamma_w  # dw/dt = 0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Phase plane
    ax = axes[0]
    ax.plot(V_nc, w_V_nullcline, "b-", linewidth=2, label="V-nullcline (dV/dt=0)")
    ax.plot(V_nc, w_w_nullcline, "r-", linewidth=2, label="w-nullcline (dw/dt=0)")
    ax.plot(V_trace, w_trace, "k-", linewidth=1, alpha=0.7, label="Trajectory")
    ax.plot(V_trace[0], w_trace[0], "go", markersize=10, label="Start")
    ax.plot(V_trace[-1], w_trace[-1], "rs", markersize=8, label="End")
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.05, 0.3)
    ax.set_xlabel("V (voltage)")
    ax.set_ylabel("w (recovery)")
    ax.set_title("Phase plane")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)

    # Time traces
    ax2 = axes[1]
    t_arr = np.linspace(0, T_total, n)
    ax2.plot(t_arr, V_trace, "C0", linewidth=2, label="V(t)")
    ax2.plot(t_arr, w_trace, "C1", linewidth=2, linestyle="--", label="w(t)")
    ax2.set_xlabel("Time (ms)")
    ax2.set_ylabel("V / w")
    ax2.set_title("Single-cell time traces")
    ax2.legend(loc="upper right")
    ax2.set_ylim(-0.3, 1.3)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    fhn_phase_plane,
    alpha_w=FloatSlider(min=0.05, max=0.45, step=0.01, value=0.10, description="α"),
    eps_w=FloatSlider(
        min=0.001,
        max=0.10,
        step=0.001,
        value=0.005,
        description="ε",
        readout_format=".3f",
    ),
    gamma_w=FloatSlider(min=0.5, max=10.0, step=0.1, value=2.0, description="γ"),
    V0=FloatSlider(min=0.0, max=0.5, step=0.01, value=0.15, description="V₀"),
);

interactive(children=(FloatSlider(value=0.1, description='α', max=0.45, min=0.05, step=0.01), FloatSlider(valu…

<function __main__.fhn_phase_plane(alpha_w=0.1, eps_w=0.005, gamma_w=2.0, V0=0.15)>

Key observations:

1. **Sub-threshold stimulus ($V_0 = 0.05$):** The trajectory stays on the left branch of the cubic V-nullcline and returns directly to the resting equilibrium without firing. The system is _excitable but not excited_.

2. **Supra-threshold stimulus ($V_0 = 0.15$):** The trajectory makes a large excursion — it jumps to the right branch of the cubic (fast upstroke), drifts upward along it as $w$ increases (plateau), falls off the right branch (repolarization), then slowly returns to rest along the bottom. This is the action potential loop.

3. **Increasing $\epsilon$:** The recovery variable $w$ responds faster, so the action potential becomes shorter and more triangular. The separation of time scales is reduced.

4. **Large $\gamma$:** The $w$-nullcline ($w = V/\gamma$) becomes nearly flat, so the resting equilibrium shifts, and the system becomes harder to excite. Small $\gamma$ makes the $w$-nullcline steeper, leading to longer action potentials and potentially sustained oscillations.
